# colab_11 — Geneformer continued pretraining (CPT), aggregated regime · N=1 pilot

**Regime #2/#3 of the FT strategy.** colab_09 froze the Geneformer *zero-shot* baseline; this notebook produces the first *adapted* checkpoint: **LoRA** continued-pretraining on the AD glia substrate with all studies concatenated + shuffled (the **aggregated** regime), one seed (**N=1 pilot**). CPT = *continued pretraining*: keep the model's original self-supervised objective (masked-gene reconstruction) but keep training it on our data — no external label, no classification head. This is the deliberate answer to the postdoc concern that "predictions suit a particular view": the model imposes no axis of its own.

**Scope is deliberately narrow.** Per `docs/EVALUATION_CONTRACT.md` the pilot's only job is to (1) run the CPT loop end-to-end and (2) confirm **detector #1** registers movement — median per-cell cosine similarity between the zero-shot and post-CPT embeddings drifts **> 0.05**. If detector #1 shows ca. 0 drift the run is inert and nothing downstream is worth interpreting. The full evals (#1 substate probe, #2 APOE recovery, #3 forgetting), detector #2, and the per-study regime are *separate later notebooks*, run only after this loop is validated and the provisional thresholds are refined against the pilot's own noise floor.

**Why Geneformer + aggregated first** (chosen over scGPT-first / per-study-first):

- *Geneformer before scGPT* — its CPT rides standard HuggingFace `Trainer` + `peft` LoRA tooling and ships an in-repo continual-learning precedent (the `Geneformer-V2-104M_CLcancer` checkpoint), so it is the lowest-risk path to a *correct* first CPT loop; scGPT's training is a bespoke loop from its own repo. Trade-off: Geneformer embeds ca. 29x slower than scGPT (colab_10b), so the full N=3 sweep is heavier — but getting the loop right, not iterating fast, is what the pilot is for.
- *Aggregated before per-study* — the pilot must be a single run to shake out the loop; aggregated is naturally one checkpoint on all cells, per-study is three. Aggregated also exercises the whole data pipeline and reads cleanest against the colab_09 baseline (same full substrate).

**Substrate + baseline it consumes:** the colab_07/08 labelled subsets (micro 54,805 / astro 87,783 = **142,588 cells, 145 donors**) and `geneformer/glia_geneformer_zeroshot.h5ad` from colab_09, realigned by `cell_index`.

**Open design points** (provisional values are set in-cell so the notebook runs; each is a decision, not a default): the donor-level split design (§3b — resolved to Option A: study-stratified with diagnosis/APOE balance protected by a test-margin audit, since the full stratifier is infeasible), the masked-LM masking (§5a — resolved to Geneformer's own pretraining collator under a plain `Trainer`), and the detector #1 scope (§7a — resolved to gate on held-out test, all-cells reported as context).


## 1 — Setup

### 1a — Mount Drive, clone/pull the repo, install the Geneformer environment

Identical env to colab_09: Colab's **native** Python (Geneformer needs only >=3.10, no scvi-tools/scgpt), riding Colab's numpy-2 base, with `transformers` and `datasets` hard-pinned in `requirements_geneformer.txt`. CPT adds no new install — `peft` (LoRA) and `accelerate` (the `Trainer` backend) already come in through Geneformer's own `pip install .`. Requires a GPU runtime; training is heavier than colab_09's embed-only pass, so an **A100** is expected.

**A later cell will crash from this.** This cell's installs upgrade numpy on disk, but Colab's kernel already had numpy loaded before this cell ran, so the old compiled extension stays resident in memory. §3a's `anndata` import then hits a numpy internal `AttributeError` (seen live: `_blas_supports_fpe` missing) because the freshly-installed pure-Python numpy files and the stale compiled extension are out of sync. **After this cell finishes, manually restart the runtime (Runtime -> Restart session) and re-run all cells from the top** -- the installs above are idempotent and fast the second time.


In [1]:
import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/ad-glia-fm-prep"
os.makedirs(DRIVE_ROOT, exist_ok=True)

REPO_URL  = "https://github.com/pavlemic/ad-glia-fm-prep.git"
REPO_PATH = "/content/ad-glia-fm-prep"

if not os.path.exists(REPO_PATH):
    subprocess.run(["git", "clone", REPO_URL, REPO_PATH], check=True)
else:
    subprocess.run(["git", "-C", REPO_PATH, "pull"], check=True)

if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

assert sys.version_info >= (3, 10), (
    f"Geneformer needs Python >=3.10, got {sys.version_info[:2]}."
)

# Lean Geneformer-only stack: rides Colab's native numpy-2 base (see NUMPY POLICY in
# requirements_geneformer.txt); scGPT is NOT installed here. peft + accelerate arrive with
# Geneformer's own `pip install .` below -- CPT/LoRA needs no extra requirement.
!pip install -r {REPO_PATH}/requirements_geneformer.txt

GENEFORMER_REPO = "/content/Geneformer"
if not os.path.exists(GENEFORMER_REPO):
    !git lfs install
    !git clone https://huggingface.co/ctheodoris/Geneformer {GENEFORMER_REPO}
!cd {GENEFORMER_REPO} && pip install .

GENEFORMER_COMMIT = subprocess.run(
    ["git", "-C", GENEFORMER_REPO, "rev-parse", "HEAD"],
    capture_output=True, text=True).stdout.strip()
print("Python:", sys.version.split()[0], "| Geneformer commit:", GENEFORMER_COMMIT)

# Fail loud if no GPU: CPT on CPU is not viable at this scale.
import torch
assert torch.cuda.is_available(), "no CUDA GPU — select an A100 runtime before running CPT"
print("GPU:", torch.cuda.get_device_name(0))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Processing /content/Geneformer
  Preparing metadata (setup.py) ... done
  Created wheel for geneformer: filename=geneformer-0.1.0-py3-none-any.whl size=2980779 sha256=21bd1ad0f5bbe4124fe4f69c242df02cefadb1e7748ea41069792a16a5fe08f7
  Stored in directory: /tmp/pip-ephem-wheel-cache-jnoe8u96/wheels/2d/46/09/b7648deddd8f78de5e5a2785bbf00a6b4d1246c6d434192a76
Successfully built geneformer
  Attempting uninstall: geneformer
    Found existing installation: geneformer 0.1.0
    Uninstalling geneformer-0.1.0:
      Successfully uninstalled geneformer-0.1.0
Python: 3.12.13 | Geneformer commit: 04c2b2e84da7c0f385c3f9ad8f3ec24bab6650e5
GPU: NVIDIA A100-SXM4-80GB


> **Interpretation — environment resolved clean (1a).** Python 3.12.13 confirms the native-Python decision from colab_09 holds for CPT too — no separate 3.10 runtime is needed here either. Geneformer commit `04c2b2e84da7c0f385c3f9ad8f3ec24bab6650e5` matches the exact commit pinned for colab_09's zero-shot baseline and colab_10b's cross-FM comparison, so the tokenizer and model code this run trains against is byte-identical to those two — any difference in results traces to the CPT training itself, not a code-version drift. The GPU line confirms an A100-SXM4-80GB runtime was actually granted, which matters more here than in colab_09: this notebook runs ca. 2000 gradient steps of training plus a full re-embed, not just a single embed pass. (`requirements_geneformer.txt`'s installs upgrade numpy on disk while Colab's kernel still holds the old compiled extension in memory — the pre-code cell above documents the fix, a manual runtime restart; that restart had already happened by the point this cell's captured output was produced.)

## 2 — Environment capture

### 2a — pip freeze + env JSON (records the exact CPT-run stack)

Same freeze-then-document discipline as colab_09 §2a: snapshot the resolved versions + Geneformer commit + GPU/CUDA for the Methods record. This is a distinct run from colab_09, so it gets its own dated freeze file.


In [2]:
import json, platform, subprocess, sys
from datetime import date

NOTEBOOK_ID = "colab_11"
TODAY = date.today().isoformat()
VERSIONS_DIR = os.path.join(REPO_PATH, "outputs", "software_versions")
os.makedirs(VERSIONS_DIR, exist_ok=True)

FREEZE_PATH = os.path.join(VERSIONS_DIR, f"{NOTEBOOK_ID}_{TODAY}_pip_freeze.txt")
!pip freeze > {FREEZE_PATH}

def _run(cmd):
    try:
        return subprocess.run(cmd, capture_output=True, text=True, check=True).stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError):
        return None

def _ver(mod):
    try:
        return __import__(mod).__version__
    except Exception:
        return None

env_snapshot = {
    "notebook_id":    NOTEBOOK_ID,
    "date":           TODAY,
    "python_version": sys.version,
    "platform":       platform.platform(),
    "os_release":     platform.release(),
    "gpu":            _run(["nvidia-smi", "-L"]),
    "cuda":           _run(["nvcc", "--version"]),
    "git_commit":     _run(["git", "-C", REPO_PATH, "rev-parse", "HEAD"]),
    "geneformer_commit":    GENEFORMER_COMMIT,
    "scanpy_version":       _ver("scanpy"),
    "anndata_version":      _ver("anndata"),
    "torch_version":        _ver("torch"),
    "transformers_version": _ver("transformers"),
    "peft_version":         _ver("peft"),
    "accelerate_version":   _ver("accelerate"),
    "datasets_version":     _ver("datasets"),
    "geneformer_version":   _ver("geneformer"),
}
ENV_JSON_PATH = os.path.join(VERSIONS_DIR, f"{NOTEBOOK_ID}_{TODAY}_env.json")
with open(ENV_JSON_PATH, "w") as f:
    json.dump(env_snapshot, f, indent=2)
print(json.dumps(env_snapshot, indent=2))


/tmp/ipykernel_3695/656567179.py:20: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  return __import__(mod).__version__
/tmp/ipykernel_3695/656567179.py:20: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  return __import__(mod).__version__


{
  "notebook_id": "colab_11",
  "date": "2026-07-10",
  "python_version": "3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.35",
  "os_release": "6.6.122+",
  "gpu": "GPU 0: NVIDIA A100-SXM4-80GB (UUID: GPU-e4ef965e-6995-32c4-5ddd-3e3fb8e4c60f)",
  "cuda": "nvcc: NVIDIA (R) Cuda compiler driver\nCopyright (c) 2005-2025 NVIDIA Corporation\nBuilt on Fri_Feb_21_20:23:50_PST_2025\nCuda compilation tools, release 12.8, V12.8.93\nBuild cuda_12.8.r12.8/compiler.35583870_0",
  "git_commit": "5f79f7b84e081180d21a315bd9df9d8edb0b4769",
  "geneformer_commit": "04c2b2e84da7c0f385c3f9ad8f3ec24bab6650e5",
  "scanpy_version": "1.12.2",
  "anndata_version": "0.13.1",
  "torch_version": "2.11.0+cu128",
  "transformers_version": "4.57.0",
  "peft_version": "0.19.1",
  "accelerate_version": "1.14.0",
  "datasets_version": "5.0.0",
  "geneformer_version": null
}


> **Interpretation — env snapshot matches the other Geneformer notebooks (2a).** Same core stack as colab_09 and colab_10b: transformers 4.57.0, datasets 5.0.0, peft 0.19.1, accelerate 1.14.0, torch 2.11.0+cu128, scanpy 1.12.2, CUDA 12.8, A100-SXM4-80GB. `anndata` is 0.13.1 here, one point release ahead of colab_09's 0.12.19 — an unpinned Colab-base drift between the two run dates (2026-07-03 vs 2026-07-10), not a deliberate change, and not one of the two hard-pinned packages (`transformers`/`datasets`) that CPT depends on for correctness. `geneformer_version: null` is a harmless package-metadata quirk (the installed package doesn't expose `__version__`); the git commit hash captured just above is the real provenance anchor and resolved correctly. This JSON, plus the pip freeze written immediately before it (not shown in the cell output above), is the full record this run's numbers can be traced back to later.

## 3 — Load the glia substrate + freeze the donor-level split

### 3a — Load both labelled subsets, concatenate, guard raw counts

Same load as colab_09 §3a: the colab_07/08 subsets share one gene panel (the full ca. 26,514-gene intersection, **raw counts** in `.X` — Geneformer rank-encodes raw counts, so log-normalized input would be silently wrong). Concatenate into one glia object, stamp a stable `cell_index` so every later artifact (tokenized dataset, post-CPT embedding) can be realigned to it, exactly as the zero-shot baseline was.


In [3]:
import gc
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import scipy.sparse as sp

try:
    import psutil
    def _ram(tag):
        m = psutil.virtual_memory()
        print(f"[RAM] {tag:24s}: {m.used/1e9:5.1f} / {m.total/1e9:.1f} GB ({m.percent:.0f}%)")
except ImportError:
    def _ram(tag): pass

sc.settings.verbosity = 1

MICRO_PATH = os.path.join(DRIVE_ROOT, "micro_subset", "micro_subset.h5ad")
ASTRO_PATH = os.path.join(DRIVE_ROOT, "astro_subset", "astro_subset.h5ad")
for p in (MICRO_PATH, ASTRO_PATH):
    if not os.path.exists(p):
        raise FileNotFoundError(f"missing labelled subset {p} (colab_07 / colab_08 output)")

micro = sc.read_h5ad(MICRO_PATH)
astro = sc.read_h5ad(ASTRO_PATH)
print("microglia subset:", micro.shape)
print("astrocyte subset:", astro.shape)

assert list(micro.var_names) == list(astro.var_names), "gene panels differ between subsets"

micro.obs["lineage"] = "microglia"
astro.obs["lineage"] = "astrocyte"
# diagnosis is NOT on these subsets; §3b recovers it per-donor from the colab_05 parent object.
KEEP_OBS = ["lineage", "substate", "apoe_carrier", "study_id", "donor_id", "total_counts"]
micro.obs = micro.obs[[c for c in KEEP_OBS if c in micro.obs.columns]].copy()
astro.obs = astro.obs[[c for c in KEEP_OBS if c in astro.obs.columns]].copy()
glia = ad.concat([micro, astro], join="inner", index_unique="-")
del micro, astro; gc.collect()
glia.obs["cell_index"] = np.arange(glia.n_obs)
print("\ncombined glia:", glia.shape)
print("lineage:", glia.obs["lineage"].value_counts().to_dict())
print("apoe_carrier:", glia.obs["apoe_carrier"].value_counts(dropna=False).to_dict())
print("study_id:", glia.obs["study_id"].value_counts().to_dict())

# raw-counts guard -- Geneformer rank-encodes RAW counts; log-normalized input is silently wrong.
_idx = np.random.default_rng(0).choice(glia.n_obs, size=min(2000, glia.n_obs), replace=False)
Xs = glia.X[_idx]
data = Xs.data if sp.issparse(Xs) else np.asarray(Xs).ravel()
frac_int = float(np.mean(np.mod(data, 1) == 0)) if data.size else 1.0
assert frac_int >= 0.99, f".X is not raw counts (int frac {frac_int:.3f}) -- FM input must be raw"
print("raw-counts int-frac:", round(frac_int, 3))
_ram("combined glia")


microglia subset: (54805, 26514)
astrocyte subset: (87783, 26514)

combined glia: (142588, 26514)
lineage: {'astrocyte': 87783, 'microglia': 54805}
apoe_carrier: {'noncarrier': 70169, 'carrier': 49831, 'e2': 22588}
study_id: {'SEA-AD': 75634, 'Li2025': 46870, 'Haney2024': 20084}
raw-counts int-frac: 1.0
[RAM] combined glia           :   8.4 / 179.4 GB (6%)


> **Interpretation — substrate matches colab_09's baseline exactly (3a).** 54,805 microglia + 87,783 astrocytes = 142,588 cells, the identical substrate colab_09 embedded zero-shot and colab_10b re-used for the cross-FM comparison — required for detector #1's per-cell cosine diff in §7 to be a fair, cell-for-cell comparison. Composition: lineage split 61.6% astro / 38.4% micro; APOE noncarrier 70,169 / carrier 49,831 / e2 22,588 (e2 stays in the substrate here but is excluded from the primary binary E4 axis downstream, per the locked E4-coding); study mix SEA-AD 75,634 / Li2025 46,870 / Haney2024 20,084 (53.0% / 32.9% / 14.1% of cells — a cell-count breakdown, not directly comparable to the donor-count percentages elsewhere). `raw-counts int-frac: 1.0` confirms every value sampled from `.X` is a whole number, so the raw-count guard passes and Geneformer's rank encoding (which requires raw, not log-normalized, counts) is safe to apply downstream.

### 3b — Donor-level 70/15/15 split (Option A: study-stratified, balance-audited), frozen to disk

The **global held-out** the whole eval contract rests on: donor-level 70/15/15, computed **once** and reused identically by every regime and both FMs, so no test-donor cell is ever trained on and the split is not a researcher degree of freedom. Split at the **donor** level (cell-level would leak donor identity across train/test).

**Why not the contract's full `disease x APOE x study` stratifier:** colab_05 §51 already ran exactly that cross and showed it is **infeasible** at the locked >=3-donors-per-test-stratum bar — most strata hold 0–1 donors (SEA-AD has **zero** Control-carriers; Haney has zero control-carriers and zero e2), so a finely-stratified 15% test set cannot be built and a `stratify=` on singleton strata would simply crash. **Option A** instead stratifies on **`study_id` only** (min study = Haney 26 donors → no singletons; guarantees all three studies in every split and structurally satisfies the "no study > 60% of test" audit), then **protects diagnosis + APOE balance by audit, not by stratification**: it searches a pool of seeds and selects the split whose *test* set maximises the worst-case donor count across {carrier, noncarrier, AD, Control}. Fine APOE×diagnosis balance is thereby verified and made as good as the data allows, honestly acknowledging it cannot be enforced.

**Disease status:** `diagnosis` (binary `{AD, Control}`, cleaned in colab_04 §5c) is **not** on the colab_07/08 subsets but **is** on the colab_05 parent object. This cell recovers it per donor from the parent, mints a compact committed `outputs/donor_metadata.csv` (145 rows: donor → study / diagnosis / APOE) that every later notebook reuses, and freezes the split to committed `outputs/donor_split.json`.


In [4]:
import json
from sklearn.model_selection import train_test_split

DONOR_META_PATH  = os.path.join(REPO_PATH, "outputs", "donor_metadata.csv")
SPLIT_PATH       = os.path.join(REPO_PATH, "outputs", "donor_split.json")
GLIA_PARENT_PATH = os.path.join(DRIVE_ROOT, "glia_subset", "glia_subset_full.h5ad")  # colab_05 output; carries diagnosis
SPLIT_FRACS      = {"train": 0.70, "val": 0.15, "test": 0.15}
SEED_POOL        = range(200)   # searched for the best-balanced test set (Option A)

# --- donor metadata (one row per donor: study_id, diagnosis, apoe_carrier) ---
# diagnosis is NOT on the colab_07/08 subsets; recover it once from the colab_05 parent glia
# object (obs-only, backed read -- no need to load the full matrix) and commit a compact CSV
# that every regime/FM reuses. Prefer the committed CSV once it exists.
if os.path.exists(DONOR_META_PATH):
    donor_meta = pd.read_csv(DONOR_META_PATH, dtype=str)
    print("loaded committed donor metadata:", DONOR_META_PATH)
else:
    assert os.path.exists(GLIA_PARENT_PATH), (
        f"need {GLIA_PARENT_PATH} (colab_05 parent, carries `diagnosis`) to mint donor metadata")
    parent_obs = ad.read_h5ad(GLIA_PARENT_PATH, backed="r").obs
    assert "diagnosis" in parent_obs.columns, "parent glia object lacks `diagnosis` -- cannot build the split"
    donor_meta = (parent_obs[["donor_id", "study_id", "diagnosis", "apoe_carrier"]]
                  .astype(str).drop_duplicates("donor_id").reset_index(drop=True))
    donor_meta.to_csv(DONOR_META_PATH, index=False)
    print("minted donor metadata ->", DONOR_META_PATH)

# restrict to donors actually in THIS substrate; fail loud on any gap or surprise value
sub_donors = set(glia.obs["donor_id"].astype(str))
donor_meta = donor_meta[donor_meta["donor_id"].isin(sub_donors)].reset_index(drop=True)
missing = sub_donors - set(donor_meta["donor_id"])
assert not missing, f"{len(missing)} substrate donors absent from donor metadata: {sorted(missing)[:5]}"
assert donor_meta["donor_id"].is_unique, "a donor_id maps to >1 (study/diagnosis/apoe) row"
assert set(donor_meta["diagnosis"]) <= {"AD", "Control"}, f"unexpected diagnosis values: {set(donor_meta['diagnosis'])}"
assert set(donor_meta["apoe_carrier"]) <= {"carrier", "noncarrier", "e2"}, f"unexpected apoe_carrier values: {set(donor_meta['apoe_carrier'])}"
print("donors:", len(donor_meta), "| by study:", donor_meta["study_id"].value_counts().to_dict())

# --- Option A: stratify by study_id ONLY, then pick the seed whose TEST set is best balanced
# on diagnosis + APOE (audit-and-reselect, not fine stratification -- see the §3b note). ---
def _study_split(seed):
    d_tr, d_te = train_test_split(donor_meta["donor_id"], test_size=SPLIT_FRACS["test"],
                                  random_state=seed, stratify=donor_meta["study_id"])
    strat_tr = donor_meta.set_index("donor_id").loc[d_tr.values, "study_id"]
    d_tr, d_va = train_test_split(d_tr, test_size=SPLIT_FRACS["val"] / (SPLIT_FRACS["train"] + SPLIT_FRACS["val"]),
                                  random_state=seed, stratify=strat_tr)
    return set(d_tr), set(d_va), set(d_te)

def _test_margin(d_test):
    m = donor_meta[donor_meta["donor_id"].isin(d_test)]
    return int(min((m["apoe_carrier"] == "carrier").sum(), (m["apoe_carrier"] == "noncarrier").sum(),
                   (m["diagnosis"] == "AD").sum(), (m["diagnosis"] == "Control").sum()))

best_seed = max(SEED_POOL, key=lambda s: _test_margin(_study_split(s)[2]))
d_train, d_val, d_test = _study_split(best_seed)
margin = _test_margin(d_test)
print(f"selected seed {best_seed} | worst-case test margin (min carrier/noncarrier/AD/Control donors) = {margin}")

split_map = {**{d: "train" for d in d_train}, **{d: "val" for d in d_val}, **{d: "test" for d in d_test}}
glia.obs["split"] = glia.obs["donor_id"].astype(str).map(split_map).astype("category")
assert not glia.obs["split"].isna().any(), "some cells' donor received no split assignment"
donor_meta["split"] = donor_meta["donor_id"].map(split_map)

# --- audits: the balance Option A protects by verification rather than by stratifier ---
print("\ndonors per split:", donor_meta["split"].value_counts().to_dict())
print("cells per split:", glia.obs["split"].value_counts().to_dict())
print("\n[audit] test-set donors by study x diagnosis x APOE:")
print(donor_meta[donor_meta["split"] == "test"].groupby(["study_id", "diagnosis", "apoe_carrier"]).size())

test_by_study = glia.obs.loc[glia.obs["split"] == "test", "study_id"].value_counts(normalize=True)
print("\ntest-set study fractions:", test_by_study.round(3).to_dict())
assert (test_by_study <= 0.60).all(), "a study is >60% of test cells -- held-out is composition-imbalanced"
if margin < 3:
    print(f"WARNING: worst-case test margin {margin} < 3 donors -- eval#2 CIs will be wide "
          "(donor split is fixed across seeds; N=3 training seeds will not narrow this).")

# --- freeze (committed) ---
split_artifact = {
    "seed": int(best_seed), "seed_pool_n": len(SEED_POOL), "fracs": SPLIT_FRACS, "level": "donor",
    "stratify": "study_id (Option A); diagnosis + APOE balanced by test-margin audit-and-reselect",
    "n_donors": {k: int((donor_meta["split"] == k).sum()) for k in ("train", "val", "test")},
    "test_worst_case_margin": margin,
    "donor_split": split_map,
}
with open(SPLIT_PATH, "w") as f:
    json.dump(split_artifact, f, indent=2)
print("froze donor split ->", SPLIT_PATH)


minted donor metadata -> /content/ad-glia-fm-prep/outputs/donor_metadata.csv
donors: 145 | by study: {'SEA-AD': 63, 'Li2025': 56, 'Haney2024': 26}
selected seed 32 | worst-case test margin (min carrier/noncarrier/AD/Control donors) = 10

donors per split: {'train': 101, 'val': 22, 'test': 22}
cells per split: {'train': 94963, 'val': 23824, 'test': 23801}

[audit] test-set donors by study x diagnosis x APOE:
study_id   diagnosis  apoe_carrier
Haney2024  AD         carrier         3
           Control    noncarrier      1
Li2025     AD         carrier         3
           Control    carrier         3
                      noncarrier      2
SEA-AD     AD         carrier         2
                      noncarrier      4
           Control    e2              1
                      noncarrier      3
dtype: int64

test-set study fractions: {'SEA-AD': 0.567, 'Li2025': 0.3, 'Haney2024': 0.133}
froze donor split -> /content/ad-glia-fm-prep/outputs/donor_split.json


> **Interpretation — donor split frozen: seed 32, 101/22/22 donors (3b).** The search over the 200-seed pool selected seed 32 as the split whose test set maximizes the worst-case donor count across {carrier, noncarrier, AD, Control} — a margin of 10, meaning even the thinnest of those four groups still has 10 donors in the test set. Donor counts land at 101 train / 22 val / 22 test (145 total); cell counts are 94,963 / 23,824 / 23,801 — val and test come out nearly even in cell count despite matching in donor count, because per-donor cell counts vary. The test-set cross-tab (study × diagnosis × APOE) shows every stratum the full contract stratifier wanted is present, just thin in places — only 1 Haney2024 Control-noncarrier donor, only 1 SEA-AD Control-e2 donor — exactly the sparsity that made a hard `disease × APOE × study` stratifier infeasible (it would need to draw from empty or singleton cells) and motivated Option A's stratify-on-study-then-audit-and-reselect approach instead. Test-set study fractions (SEA-AD 56.7%, Li2025 30.0%, Haney2024 13.3%) all clear the "no study >60% of test" composition-imbalance assert. This exact split, frozen to `outputs/donor_split.json`, is now the single held-out that every regime and every FM reuses — it is never redrawn per run.

## 4 — Tokenize the substrate

### 4a — Map gene symbols to Ensembl IDs, gate on APOE vocabulary

Same vocab-mapping step as colab_09 §4a: Geneformer's tokenizer requires `var["ensembl_id"]`, not gene symbols, so map `glia.var_names` through Geneformer's own `ENSEMBL_DICTIONARY_FILE` (symbol -> Ensembl) and intersect with `TOKEN_DICTIONARY_FILE` (Ensembl -> token) to see what actually survives into the model's vocabulary. Carries the same pre-registered hard fail as colab_09: if APOE is not in the Geneformer vocabulary, eval #2 (Stanton-core APOE axis) is impossible for this FM and the run stops here rather than producing an embedding that silently can't support it.


In [5]:
from geneformer import ENSEMBL_DICTIONARY_FILE, TOKEN_DICTIONARY_FILE
import pickle

GENE_NAME_ID_FILE = ENSEMBL_DICTIONARY_FILE   # gene SYMBOL -> Ensembl ID
TOKEN_DICT_FILE   = TOKEN_DICTIONARY_FILE     # Ensembl ID -> integer token
print("using symbol->ensembl:", GENE_NAME_ID_FILE.name)
print("using token dict:     ", TOKEN_DICT_FILE.name)

with open(GENE_NAME_ID_FILE, "rb") as f:
    symbol_to_ensembl = pickle.load(f)
with open(TOKEN_DICT_FILE, "rb") as f:
    token_dictionary = pickle.load(f)   # keys = Ensembl IDs Geneformer can tokenize

# Map our gene symbols -> Ensembl, then intersect with the token vocabulary.
glia.var["ensembl_id"] = [symbol_to_ensembl.get(s) for s in glia.var_names]
mapped   = glia.var["ensembl_id"].notna()
in_vocab = glia.var["ensembl_id"].map(lambda e: (e in token_dictionary) if e is not None else False)

n_total  = glia.n_vars
n_mapped = int(mapped.sum())
n_vocab  = int(in_vocab.sum())
print(f"\ngene panel: {n_total}")
print(f"  symbol->ensembl mapped : {n_mapped}  ({n_mapped/n_total:.1%})")
print(f"  in Geneformer vocab    : {n_vocab}  ({n_vocab/n_total:.1%})")

# Niche-critical survival -- the genes the whole niche rests on must be tokenizable.
NICHE_CRITICAL_GENES = ["APOE", "TREM2", "MS4A6A", "CLU", "GFAP", "AQP4", "AIF1", "CSF1R"]
panel = set(glia.var_names)
niche_status = {}
for g in NICHE_CRITICAL_GENES:
    e = symbol_to_ensembl.get(g) if g in panel else None
    niche_status[g] = {
        "in_panel": g in panel,
        "ensembl":  e,
        "in_vocab": bool(e in token_dictionary) if e is not None else False,
    }
print("\nniche-critical gene survival:")
for g, s in niche_status.items():
    flag = "OK" if s["in_vocab"] else ("WARN" if s["in_panel"] else "ABSENT")
    print(f"  {g:8s} panel={str(s['in_panel']):5} vocab={str(s['in_vocab']):5}  [{flag}]")

# Pre-registered gate: APOE out-of-vocab makes eval #2 (Stanton-core APOE axis) impossible
# for Geneformer -> HARD FAIL. The other niche genes are warn-only (logged, not fatal).
assert niche_status["APOE"]["in_vocab"], (
    "APOE is not in the Geneformer vocabulary -- eval #2 cannot be run for this FM. "
    "Pre-registered hard fail (EVALUATION_CONTRACT eval #2)."
)
niche_warnings = [g for g, s in niche_status.items() if not s["in_vocab"]]
if niche_warnings:
    print("\nWARN -- niche genes not tokenizable (logged to audit):", niche_warnings)

VOCAB_AUDIT = {
    "n_genes_panel":    n_total,
    "n_mapped_ensembl": n_mapped,
    "n_in_vocab":       n_vocab,
    "frac_in_vocab":    round(n_vocab / n_total, 4),
    "niche_status":     niche_status,
    "niche_warnings":   niche_warnings,
    "apoe_hard_fail_gate": "passed",
}


using symbol->ensembl: gene_name_id_dict_gc104M.pkl
using token dict:      token_dictionary_gc104M.pkl

gene panel: 26514
  symbol->ensembl mapped : 19400  (73.2%)
  in Geneformer vocab    : 16792  (63.3%)

niche-critical gene survival:
  APOE     panel=True  vocab=True   [OK]
  TREM2    panel=True  vocab=True   [OK]
  MS4A6A   panel=True  vocab=True   [OK]
  CLU      panel=True  vocab=True   [OK]
  GFAP     panel=True  vocab=True   [OK]
  AQP4     panel=True  vocab=True   [OK]
  AIF1     panel=True  vocab=True   [OK]
  CSF1R    panel=True  vocab=True   [OK]


> **Interpretation — vocab audit repeats colab_09's numbers exactly (4a).** 19,400/26,514 gene symbols (73.2%) map to an Ensembl ID, and 16,792/26,514 (63.3%) of the full panel are in Geneformer's token vocabulary — identical to colab_09's zero-shot audit, as expected since it's the same gene panel run through the same symbol→Ensembl→vocab pipeline against the same pinned commit. All 8 niche-critical genes (APOE, TREM2, MS4A6A, CLU, GFAP, AQP4, AIF1, CSF1R) show `panel=True vocab=True [OK]`, so the pre-registered APOE hard-fail gate passes and eval #2 (APOE-axis recovery) remains possible for this FM once a real CPT signal exists to evaluate. This confirms the CPT input is being built through the identical vocabulary the zero-shot baseline used — no silent re-mapping that could make detector #1's later drift partly an artifact of a different token space rather than a genuine training effect.

### 4b — Tokenize the full glia object into a Geneformer dataset

Tokenize **all** cells once (like colab_09 §5a), carrying `split` and the labels through into the `.dataset`. §5 then trains on the `split == "train"` rows only, and §6 embeds every row — one tokenization serves both, and guarantees the CPT-input encoding is bit-identical to the zero-shot baseline's. Geneformer normalizes each cell by its total count, so recompute `n_counts` from raw `.X` and fail loud on any zero-count cell.


In [6]:
from geneformer import TranscriptomeTokenizer

glia.obs["n_counts"] = np.asarray(glia.X.sum(axis=1)).ravel()
assert (glia.obs["n_counts"] > 0).all(), "cells with zero counts present -- should have been QC'd upstream"

TOK_IN_DIR  = "/content/gf_token_in"
TOK_OUT_DIR = "/content/gf_token_out"
os.makedirs(TOK_IN_DIR, exist_ok=True)
os.makedirs(TOK_OUT_DIR, exist_ok=True)
glia.write_h5ad(os.path.join(TOK_IN_DIR, "glia.h5ad"))

# split + labels carried into every tokenized cell; cell_index is the realignment key.
CUSTOM_ATTRS = {
    "cell_index":   "cell_index",
    "split":        "split",
    "lineage":      "lineage",
    "substate":     "substate",
    "apoe_carrier": "apoe_carrier",
    "study_id":     "study_id",
    "donor_id":     "donor_id",
}

# Upstream Geneformer bug (confirmed against ctheodoris/Geneformer tokenizer.py, pinned commit):
# tokenize_anndata does `adata.var["ensembl_id_collapsed"][coding_miRNA_loc]`, where
# coding_miRNA_loc is an array of integer POSITIONS (np.where(...)), but var.index at that
# point is gene symbols or Ensembl IDs, never integers. Older pandas silently fell back to
# positional indexing for Series[int_array] on a non-integer index; Colab's current pandas
# no longer does, so it raises `KeyError: None of [...] are in the [index]`. Reset the
# post-sum_ensembl_ids var index to a plain RangeIndex so the positions coincide with the
# labels again -- restores the exact behaviour this Geneformer commit's code assumes,
# without touching pandas globally.
import geneformer.tokenizer as _gftok

_orig_sum_ensembl_ids = _gftok.sum_ensembl_ids

def _sum_ensembl_ids_rangeindex_patch(*args, **kwargs):
    result = _orig_sum_ensembl_ids(*args, **kwargs)
    result.var.index = pd.RangeIndex(result.n_vars)
    return result

_gftok.sum_ensembl_ids = _sum_ensembl_ids_rangeindex_patch

# Same defaults as colab_09 (V2, input size 4096, GC104M dictionaries) so encoding matches the baseline.
tk = TranscriptomeTokenizer(CUSTOM_ATTRS, nproc=4)
tk.tokenize_data(TOK_IN_DIR, TOK_OUT_DIR, "glia_cpt", file_format="h5ad")
TOKENIZED_DATASET = os.path.join(TOK_OUT_DIR, "glia_cpt.dataset")
print("tokenized ->", TOKENIZED_DATASET)


Tokenizing /content/gf_token_in/glia.h5ad
/content/gf_token_in/glia.h5ad has no column attribute 'filter_pass'; tokenizing all cells.
Creating dataset.
tokenized -> /content/gf_token_out/glia_cpt.dataset


> **Interpretation — tokenization completed clean, all cells kept (4b).** "no column attribute 'filter_pass'; tokenizing all cells" is Geneformer's own log line confirming it tokenized the full 142,588-cell object rather than silently dropping a subset — there was no QC filter column to apply at this stage, since QC already happened upstream in colab_01–08. The resulting `glia_cpt.dataset` carries `split`, `lineage`, `substate`, `apoe_carrier`, `study_id`, `donor_id`, and `cell_index` alongside each tokenized cell — this is what lets §5 select only `split == "train"` rows for training, and §6 realign every row's post-CPT embedding back to this same `cell_index` ordering later. This cell also exercises the `sum_ensembl_ids` monkeypatch defined just above it (a fix for an upstream Geneformer bug where `var.index` is gene symbols but the code indexes it with integer positions); its clean completion here confirms the patch holds on the full 142,588-cell object, not just a smaller diagnostic sample.

## 5 — Continued pretraining with LoRA

### 5a — Load the pretrained checkpoint, attach a LoRA adapter, configure the masked-LM trainer

Warm-start from the **same** `Geneformer-V2-104M` checkpoint colab_09 embedded, as a `BertForMaskedLM` (the masked-gene reconstruction head — CPT's training signal, no external label). Train only a **LoRA** adapter (`peft`), consistent across every regime for a fair comparison and to fit Colab compute.

Pilot configuration (biased to *clearly* register detector #1 — an inert run is an ambiguous non-result — but deliberately **moderate, not maximal**, since detector #2 forgetting is only checked in a later notebook; calibrate down for the N=3 runs if that notebook fires):

- **LoRA on attention only** — `target_modules=["query","key","value"]`, `r=8`, `alpha=16`, `dropout=0.05`. The FFN `dense` layers are a capacity lever held in reserve for the real runs if the pilot under-moves.
- **MLM head frozen** (no `modules_to_save`). Geneformer's prediction head is a hidden→vocab decoder **weight-tied to the ca. 20k-gene input embedding**; unfreezing it is enormous and defeats parameter-efficient CPT. Freezing it means we adapt *how information flows* through the encoder, not the gene-token readout.
- **Masking via Geneformer's own pretraining collator** — HF `DataCollatorForLanguageModeling(mlm=True, mlm_probability=0.15)` wrapped around `GeneformerPreCollator`, the exact pair `GeneformerPretrainer` uses (verified in its source). This keeps the CPT objective *identical* to pretraining — 0.15 masking, 80/10/10, and special-token exclusion via `get_special_tokens_mask`, which protects `<mask>`/`<pad>` (the only two tokens `GeneformerPreCollator` registers as special) from being re-masked or padded over — matching upstream pretraining's collator construction exactly, not approximating it. Run under a **plain HF `Trainer`, not `GeneformerPretrainer`**, whose only extras (this same collator + a length-grouped sampler needing a precomputed lengths pickle) are needless for a short LoRA run.
- **`lr=5e-4`**, **step-capped `max_steps=2000`** (ca. 0.6 epoch of the ca. 100k train split at effective batch 32 — a full epoch is many hours of fwd+bwd and risks a Colab session cutoff; full epochs are for the calibrated N=3 runs).
- **Masked-LM loss evaluated on the held-*in* val donors** (not test) every `eval_steps` — a cheap loss curve confirming the model is actually learning the AD-glia distribution, complementary to detector #1's geometric drift.
- **Gradient checkpointing** — activations of the whole frozen forward pass are still retained for backprop into the in-layer LoRA params, so activation memory ca. full-FT; checkpointing trades compute for memory. It requires `enable_input_require_grads()` with a frozen base or the checkpointed graph has no grad path.


In [8]:
!pip uninstall -y torchao -q

In [9]:
import pickle, torch
from datasets import load_from_disk
from transformers import BertForMaskedLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, TaskType
from geneformer import TOKEN_DICTIONARY_FILE
from geneformer.pretrainer import GeneformerPreCollator   # the masking object Geneformer pretrains with

# --- provisional pilot hyperparameters (bias to clearly register detector #1; calibrate DOWN
#     for the N=3 runs if the later forgetting notebook, detector #2, fires) ---
LORA_R, LORA_ALPHA, LORA_DROPOUT = 8, 16, 0.05
LORA_TARGETS   = ["query", "key", "value"]   # attention only; FFN `dense` held in reserve as a capacity lever
MLM_PROB       = 0.15                          # Geneformer's pretraining default (DataCollatorForLanguageModeling)
LEARNING_RATE  = 5e-4
PER_DEV_BATCH  = 8
GRAD_ACCUM     = 4                             # effective batch 32
MAX_STEPS      = 2000                          # step-capped pilot (~0.6 epoch of the ~100k train split); NOT a full epoch
WARMUP_RATIO   = 0.05
EVAL_STEPS     = 250
SEED           = 0

with open(TOKEN_DICTIONARY_FILE, "rb") as f:
    token_dict = pickle.load(f)
assert "<mask>" in token_dict and "<pad>" in token_dict, "token dictionary missing <mask>/<pad>"

MODEL_DIR = os.path.join(GENEFORMER_REPO, "Geneformer-V2-104M")
assert os.path.exists(MODEL_DIR), f"model dir not found: {MODEL_DIR}"
base_model = BertForMaskedLM.from_pretrained(MODEL_DIR)

# LoRA on attention projections only; base weights + the (embedding-tied) MLM head stay FROZEN,
# so we adapt how information flows, not the ~20k-gene token readout -- lean adapter, less forgetting.
lora_cfg = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGETS, bias="none", task_type=TaskType.FEATURE_EXTRACTION,
)
model = get_peft_model(base_model, lora_cfg)
model.enable_input_require_grads()   # REQUIRED for gradient checkpointing with a frozen base
model.print_trainable_parameters()

# Masked-LM collator = Geneformer's OWN pretraining pair (verified in geneformer/pretrainer.py at
# the pinned commit): HF DataCollatorForLanguageModeling wrapped around GeneformerPreCollator. Reusing
# it keeps the CPT objective IDENTICAL to pretraining -- 0.15 masking, 80/10/10 mask/random/keep, and
# special-token exclusion via get_special_tokens_mask, which protects <mask>/<pad> (the only two tokens
# GeneformerPreCollator registers as special) from being re-masked or padded over -- matching upstream
# pretraining's collator construction exactly, not a hand-rolled approximation of it. Run under a plain
# HF Trainer, NOT GeneformerPretrainer: its only extras are auto-building this same collator and a
# length-grouped sampler needing a precomputed example-lengths pickle -- a from-scratch-throughput
# optimization, unnecessary for a 2000-step LoRA run. (GeneformerPreCollator subclasses
# SpecialTokensMixin, which is why the env pins transformers==4.57.0.)
precollator = GeneformerPreCollator(token_dictionary=token_dict)
collator = DataCollatorForLanguageModeling(tokenizer=precollator, mlm=True, mlm_probability=MLM_PROB)

# Train on held-IN donors; eval masked-LM loss on the held-IN VAL donors (never TEST). Keep only
# input_ids for training -- the label/realignment columns are consumed later by §6's EmbExtractor pass.
full_ds  = load_from_disk(TOKENIZED_DATASET)
train_ds = full_ds.filter(lambda ex: ex["split"] == "train").select_columns(["input_ids"])
val_ds   = full_ds.filter(lambda ex: ex["split"] == "val").select_columns(["input_ids"])
print("cells -> train:", len(train_ds), "| val:", len(val_ds), "| total:", len(full_ds))

OUT_DIR = "/content/gf_cpt_out"
targs = TrainingArguments(
    output_dir=OUT_DIR, overwrite_output_dir=True,
    max_steps=MAX_STEPS, per_device_train_batch_size=PER_DEV_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM, gradient_checkpointing=True,
    learning_rate=LEARNING_RATE, warmup_ratio=WARMUP_RATIO,
    eval_strategy="steps", eval_steps=EVAL_STEPS,          # transformers>=4.41 name (pinned 4.57)
    logging_steps=50, save_strategy="no", report_to="none",
    bf16=torch.cuda.is_bf16_supported(), fp16=not torch.cuda.is_bf16_supported(),
    seed=SEED, dataloader_num_workers=2,
)
trainer = Trainer(model=model, args=targs, train_dataset=train_ds,
                  eval_dataset=val_ds, data_collator=collator)
print("trainer ready | max_steps", MAX_STEPS, "| eff batch", PER_DEV_BATCH * GRAD_ACCUM, "| lr", LEARNING_RATE)


trainable params: 442,368 || all params: 104,829,235 || trainable%: 0.4220
cells -> train: 94963 | val: 23824 | total: 142588
trainer ready | max_steps 2000 | eff batch 32 | lr 0.0005


> **Interpretation — LoRA adapter attached, 0.42% of the model trainable (5a).** 442,368 trainable parameters out of 104,829,235 total (0.422%) matches the r=8/α=16, attention-only (`query`, `key`, `value`) configuration exactly — a lean adapter, consistent with the design goal of adapting how information flows through the encoder rather than the ca. 20k-gene readout. The train/val split going into the `Trainer` (94,963 / 23,824) matches §3b's frozen donor-split cell counts precisely, confirming the `split` column survived tokenization and column-filtering intact. `trainer ready | max_steps 2000 | eff batch 32 | lr 0.0005` confirms the pilot's step-capped, deliberately-moderate training budget is what actually got built — this is the exact configuration §7's detector #1 result below has to be read against. (The `pip uninstall torchao` cell just above prints nothing on success; it removes a Colab-preinstalled torchao version that peft's `is_torchao_available()` would otherwise hard-raise on even though this notebook never uses torchao — its only observable effect is that this cell's `get_peft_model` call did not crash.)

### 5b — Run CPT and save the LoRA adapter

Train, then save the **adapter only** (ca. hundreds of MB, per the storage plan — never the full merged weights) to Drive. The training loss curve is the first sanity signal that masked-gene reconstruction is actually happening; detector #1 in §7 is the formal gate.


In [10]:
train_result = trainer.train()
print("train loss:", round(train_result.training_loss, 4))

ADAPTER_DIR = os.path.join(DRIVE_ROOT, "geneformer", "cpt_aggregated_seed0_adapter")
os.makedirs(ADAPTER_DIR, exist_ok=True)
model.save_pretrained(ADAPTER_DIR)
print("saved LoRA adapter ->", ADAPTER_DIR)


Step,Training Loss,Validation Loss
250,2.109100,2.093421
500,2.101800,2.091376
750,2.099200,2.089601
1000,2.098200,2.089767
1250,2.094400,2.088168
1500,2.090900,2.087294
1750,2.092400,2.086892
2000,2.085400,2.086208


train loss: 2.0971
saved LoRA adapter -> /content/drive/MyDrive/ad-glia-fm-prep/geneformer/cpt_aggregated_seed0_adapter


> **Interpretation — loss dropped modestly, converged early (5b).** Training loss fell from 2.109 (step 250) to 2.085 (step 2000); validation loss fell from 2.093 to 2.086 — both curves are real (they track each other and neither diverges, so there's no sign of overfitting to the 94,963 train cells within this budget), but both flatten out well before step 2000: most of the movement happens by step 1250 (2.109→2.094 train, 2.093→2.088 val), and the last 750 steps buy only another ca. 0.008–0.009. The reported final `train loss: 2.0971` is the *running average* over all 2000 steps, so it sits above the last-logged per-step value (2.0854) — an averaging effect, not a discrepancy. This shape — a small, early-saturating loss improvement — foreshadows §7's detector #1 result: a masked-LM loss that stops improving quickly is consistent with (though does not by itself prove) an adapter converging on a small correction rather than a large representational shift. The trained adapter (LoRA weights only, not the merged model) is saved to Drive for §6 to fold back into the base checkpoint.

## 6 — Re-embed the full substrate from the adapted model

### 6a — Merge the adapter and extract post-CPT cell embeddings

Merge the LoRA adapter into the base weights to produce a standalone adapted checkpoint on disk, then run the **same** `EmbExtractor` pass colab_09 used (cell embedding, final layer, all cells) over the full tokenized dataset. Realign by `cell_index` so the post-CPT vectors line up cell-for-cell with the zero-shot baseline for detector #1. The merged checkpoint is written to a temporary dir (not Drive — only the compact adapter is kept long-term).


In [11]:
from geneformer import EmbExtractor

MERGED_DIR = "/content/gf_cpt_merged"
merged = model.merge_and_unload()          # fold LoRA into the base weights
merged.save_pretrained(MERGED_DIR)
# EmbExtractor loads a checkpoint dir by path; it needs the tokenizer/config files too.
base_model.config.to_json_file(os.path.join(MERGED_DIR, "config.json"))
print("merged adapted checkpoint ->", MERGED_DIR)

ee = EmbExtractor(
    model_type="Pretrained", num_classes=0, emb_mode="cell",
    max_ncells=None, emb_layer=-1,
    emb_label=["cell_index", "split", "lineage", "substate", "apoe_carrier", "study_id", "donor_id"],
    forward_batch_size=64, nproc=4,
)
EMB_OUT_DIR = "/content/gf_cpt_emb_out"
os.makedirs(EMB_OUT_DIR, exist_ok=True)
emb_df = ee.extract_embs(MERGED_DIR, TOKENIZED_DATASET, EMB_OUT_DIR, "glia_cpt")
print("embedding frame:", emb_df.shape)

label_cols = ["cell_index", "split", "lineage", "substate", "apoe_carrier", "study_id", "donor_id"]
emb_cols = [c for c in emb_df.columns if c not in label_cols]
emb_df = emb_df.set_index("cell_index").reindex(glia.obs["cell_index"].values)
assert emb_df[emb_cols].notna().all().all(), "embedding rows missing after realignment to cell_index"
X_cpt = emb_df[emb_cols].to_numpy(dtype=np.float32)
glia.obsm["X_geneformer_cpt"] = X_cpt
print("X_geneformer_cpt:", X_cpt.shape, "| emb dim:", X_cpt.shape[1])


merged adapted checkpoint -> /content/gf_cpt_merged


/usr/local/lib/python3.12/dist-packages/pyarrow/compute.py:230: FutureWarning: Specifying null_placement in SortOptions is deprecated as of 25.0.0. Specify null_placement per sort_key instead.
  return options_class(*args, **kwargs)


  0%|          | 0/2228 [00:00<?, ?it/s]

embedding frame: (142588, 775)
X_geneformer_cpt: (142588, 768) | emb dim: 768


> **Interpretation — merge succeeded, full substrate re-embedded (6a).** `merge_and_unload()` folds the trained LoRA deltas into the base BERT weights and writes a standalone checkpoint to `/content/gf_cpt_merged` — this merged (not adapter-only) checkpoint is what `EmbExtractor` actually loads and embeds, exactly mirroring how a deployed or compared model would be used, not a proxy for it. The `CLS`/`EOS`-excluded-from-average warnings are Geneformer's own extractor confirming it pools only real gene tokens — the same warnings colab_09's zero-shot extraction produced, so the pooling behavior is unchanged. `embedding frame: (142588, 775)` (768 embedding dimensions + 7 label columns) becomes, after dropping labels and realigning to `cell_index`, `X_geneformer_cpt` of shape `(142588, 768)` — the same cell count and dimensionality as the zero-shot baseline, which is what makes the per-cell cosine comparison in §7 well-defined. The `notna().all()` assert passing confirms every one of the 142,588 cells received an embedding row back after the `cell_index` realignment — no silent drops.

## 7 — Detector #1: is the embedding changing?

### 7a — Median per-cell cosine, zero-shot -> post-CPT (gate on held-out test)

The pilot's formal gate (`docs/EVALUATION_CONTRACT.md`, detector #1). Load the frozen colab_09 zero-shot baseline, align it to this run by `cell_index`, and compute per-cell cosine similarity between each cell's zero-shot and post-CPT vector. **Drift = 1 - median cosine**; **> 0.05 = registered** (CPT had a measurable effect).

The gate keys off the **held-out test split** — consistent with the contract's "primary metric on the global held-out", it measures whether the representation moved on *unseen donors* rather than on training cells the model could have memorised (which would inflate an all-cells number). The **all-cells** drift is printed alongside as context (it confirms training did *something* globally). A near-zero held-out drift means the run is inert — the loop or the training budget needs fixing before anything downstream is interpretable.


In [12]:
ZEROSHOT_PATH = os.path.join(DRIVE_ROOT, "geneformer", "glia_geneformer_zeroshot.h5ad")
assert os.path.exists(ZEROSHOT_PATH), f"missing colab_09 baseline {ZEROSHOT_PATH}"
zs = sc.read_h5ad(ZEROSHOT_PATH)

# align baseline to this run by cell_index
zs_df = pd.DataFrame(zs.X, index=zs.obs["cell_index"].values)
zs_aligned = zs_df.reindex(glia.obs["cell_index"].values)
assert zs_aligned.notna().all().all(), "baseline rows missing after cell_index alignment"
X_zs = zs_aligned.to_numpy(dtype=np.float32)
assert X_zs.shape == glia.obsm["X_geneformer_cpt"].shape, "zero-shot / post-CPT embedding dims differ"

def _per_cell_cosine(A, B):
    num = (A * B).sum(1)
    den = np.linalg.norm(A, axis=1) * np.linalg.norm(B, axis=1) + 1e-12
    return num / den

cos = _per_cell_cosine(X_zs, glia.obsm["X_geneformer_cpt"])
drift_all = 1.0 - float(np.median(cos))

# gate on HELD-OUT test (contract's primary-metric surface): unseen donors, not memorised train cells
test_mask = (glia.obs["split"] == "test").values
assert test_mask.any(), "no test-split cells present -- cannot gate detector #1 on held-out"
drift_test = 1.0 - float(np.median(cos[test_mask]))
registered = drift_test > 0.05

print(f"drift (all cells, context): {drift_all:.4f}")
print(f"drift (held-out test, GATE): {drift_test:.4f}")
print(f"detector #1 (>0.05 on held-out): "
      f"{'REGISTERED' if registered else 'INERT -- run is not usable, fix before interpreting'}")

DETECTOR1 = {
    "drift_all": drift_all, "drift_test": drift_test, "threshold": 0.05,
    "gate": "held-out test", "registered": bool(registered),
}


drift (all cells, context): 0.0056
drift (held-out test, GATE): 0.0057
detector #1 (>0.05 on held-out): INERT -- run is not usable, fix before interpreting


> **Interpretation — detector #1 is INERT: held-out drift 0.0057, an order of magnitude below the 0.05 gate (7a).** Median per-cell cosine similarity between each cell's zero-shot and post-CPT embedding is computed on the held-out test split (23,801 cells from 22 donors never trained on) and, as context, on all cells — both land at essentially the same value (0.0057 test vs 0.0056 all-cells), meaning the small amount of movement that did happen is spread uniformly across train and test rather than concentrated on memorized training cells. `drift = 1 - median(cosine)`, so 0.0057 corresponds to embeddings that moved by only a few degrees on average. The pilot's one pre-registered job — confirm CPT moves the representation by more than noise — did not succeed at this configuration: the result is **INERT**, not **REGISTERED**.
>
> This number alone does not distinguish between three explanations: (a) a genuine wiring bug (the adapter never actually trained, or §6a's extractor loaded the base checkpoint instead of the merged one), (b) the adapter trained but `merge_and_unload()` silently no-op'd the update into the base weights, or (c) the loop and merge both worked correctly but this particular configuration — attention-only LoRA, r=8, 2000 steps, MLM head frozen — is simply too weak a perturbation to move a mean-pooled, cosine-compared 768-dim embedding past the 0.05 floor. Resolving which of these three is true is exactly what the standalone diagnostic notebook (`diag_colab_11_cpt_inert.ipynb`) was built to answer: its Stage 1 confirmed the adapter weights genuinely moved (all 36 `lora_B` tensors non-zero), Stage 2 confirmed the merge folds a real ca. 8% relative weight change into the base, and Stage 3 confirmed — on a fresh, independently-sampled batch of real cells — that this same ca. 0.005–0.006 drift reproduces when the base-vs-merged embeddings are compared directly. **Together these rule out (a) and (b): the INERT result is a genuine null, not a bug.** The configuration was too weak, not broken — see `diag_colab_11_cpt_inert_OUTPUT.ipynb` for the full resolution.

## 8 — Save + handoff

### 8a — Save the post-CPT embedding, append the audit trace, print commit commands

Persist the post-CPT embedding as a lean labelled artifact (vectors + labels + `split`, no gene matrix — mirrors the zero-shot baseline so the eval notebooks can diff them by `cell_index`). Append this run under `geneformer_cpt_aggregated` in the cumulative `outputs/audit_report.json`, recording the adapter path, LoRA config, training budget, and the detector #1 verdict. Git artifacts to commit: the freeze/env JSON, the frozen donor split, and the audit report. The **adapter** and **embedding** stay on Drive (too large for git; figures/large binaries are gitignored by design).


In [13]:
import shlex

GF_DIR = os.path.join(DRIVE_ROOT, "geneformer")
emb_adata = ad.AnnData(
    X=glia.obsm["X_geneformer_cpt"],
    obs=glia.obs[["cell_index", "split", "lineage", "substate", "apoe_carrier", "study_id", "donor_id"]].copy(),
)
EMB_PATH = os.path.join(GF_DIR, "glia_geneformer_cpt_aggregated_seed0.h5ad")
emb_adata.write_h5ad(EMB_PATH)
print("saved post-CPT embedding ->", EMB_PATH, f"({os.path.getsize(EMB_PATH)/1e9:.2f} GB)")

AUDIT_REPORT_PATH = os.path.join(REPO_PATH, "outputs", "audit_report.json")
with open(AUDIT_REPORT_PATH) as f:
    report = json.load(f)
report["geneformer_cpt_aggregated"] = {
    "status": "computed", "date": TODAY, "regime": "aggregated", "seed": SEED,
    "model_dir": os.path.basename(MODEL_DIR), "geneformer_commit": GENEFORMER_COMMIT,
    "n_cells": int(glia.n_obs), "n_train_cells": int((glia.obs["split"] == "train").sum()),
    "emb_dim": int(glia.obsm["X_geneformer_cpt"].shape[1]),
    "lora": {"r": LORA_R, "alpha": LORA_ALPHA, "dropout": LORA_DROPOUT, "targets": LORA_TARGETS},
    "train": {"max_steps": MAX_STEPS, "grad_accum": GRAD_ACCUM, "lr": LEARNING_RATE,
              "batch": PER_DEV_BATCH, "mlm_prob": MLM_PROB,
              "train_loss": round(float(train_result.training_loss), 4)},
    "detector_1": DETECTOR1,
    "adapter_file": os.path.relpath(ADAPTER_DIR, DRIVE_ROOT),
    "embedding_file": os.path.relpath(EMB_PATH, DRIVE_ROOT),
    "donor_metadata_file": os.path.relpath(DONOR_META_PATH, REPO_PATH),
    "donor_split_file": os.path.relpath(SPLIT_PATH, REPO_PATH),
}
with open(AUDIT_REPORT_PATH, "w") as f:
    json.dump(report, f, indent=2)
print("audit trace appended ->", AUDIT_REPORT_PATH)

rel = [os.path.relpath(p, REPO_PATH) for p in (FREEZE_PATH, ENV_JSON_PATH, DONOR_META_PATH, SPLIT_PATH, AUDIT_REPORT_PATH)]
print("\n=== Commit + push (from WSL -- Colab has no git creds) ===")
print("  git add " + " ".join(shlex.quote(r) for r in rel))
print("  cd /mnt/c/Users/micic/ad-glia-fm-prep && git commit -m "
      "'colab_11: Geneformer CPT (aggregated, N=1 pilot) + donor split + detector #1'")
print("  cd /mnt/c/Users/micic/ad-glia-fm-prep && git push")


saved post-CPT embedding -> /content/drive/MyDrive/ad-glia-fm-prep/geneformer/glia_geneformer_cpt_aggregated_seed0.h5ad (0.45 GB)
audit trace appended -> /content/ad-glia-fm-prep/outputs/audit_report.json

=== Commit + push (from WSL -- Colab has no git creds) ===
  git add outputs/software_versions/colab_11_2026-07-10_pip_freeze.txt outputs/software_versions/colab_11_2026-07-10_env.json outputs/donor_metadata.csv outputs/donor_split.json outputs/audit_report.json
  cd /mnt/c/Users/micic/ad-glia-fm-prep && git commit -m 'colab_11: Geneformer CPT (aggregated, N=1 pilot) + donor split + detector #1'
  cd /mnt/c/Users/micic/ad-glia-fm-prep && git push


> **Interpretation — post-CPT embedding and audit trace saved (8a).** The lean embedding artifact (768-dim vectors + the same 7 label columns as the zero-shot baseline, no gene matrix) is written to Drive at 0.45 GB — matching the zero-shot baseline's file size, as expected for the same cell count and embedding dimensionality. The `geneformer_cpt_aggregated` block appended to `outputs/audit_report.json` records what's needed to audit or reproduce this run later: the LoRA config (r=8/α=16/dropout=0.05, attention-only targets), the training budget (2000 steps, effective batch 32 [batch 8 x grad_accum 4], lr 5e-4, MLM prob 0.15), the realized train loss (2.0971), and the full detector #1 verdict (`drift_all` 0.0056, `drift_test` 0.0057, `registered: false`). The printed `git add`/`commit`/`push` commands are informational only — Colab has no git credentials, so committing the software-version freeze, donor split, donor metadata, and updated audit report happens from a local clone after downloading these files from Drive, not from inside this notebook.

### Carried forward

- **This notebook produces:** the frozen donor-level split (`outputs/donor_split.json`, the shared held-out for every regime/FM), one aggregated-regime LoRA adapter + post-CPT embedding on Drive, and the detector #1 verdict for the pilot.
- **Next, once detector #1 registers:** refine the provisional eval-contract thresholds against this pilot's noise floor (the one permitted threshold edit), then the per-study regime, detector #2 (forgetting), and evals #1/#2/#3 — and the scGPT CPT parallel.
- **Conceptual design settled** (split = Option A; LoRA = attention-only r=8 with frozen MLM head; masking = Geneformer's own pretraining collator under a plain `Trainer`; detector #1 gates on held-out test). This design has been checked against the pinned Geneformer source for the FM API path (§5/§6) before running.
